In [1]:
# Load environment and suppress warnings

import os
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

Key loaded: True


In [2]:
# Import LangChain components and initialize deterministic model
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

if os.getenv("OPENAI_API_KEY"):
    model = ChatOpenAI(model="gpt-5-mini", temperature=0)
else:
    model = None
    print("Skipping model init. No API key found.")

In [3]:
# Define the typed schema that represents the output contract

from enum import Enum
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Priority(str, Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"

class Ticket(BaseModel):
    priority: Priority = Field(description="Priority of the ticket: Low, Medium, or High")
    issue: str = Field(description="Short description of the problem")
    action: str = Field(description="Next action to take")

parser = PydanticOutputParser(pydantic_object=Ticket)
format_instructions = parser.get_format_instructions()

In [4]:
# Create one sample ticket text that we will reuse across both chains

ticket_text = """
Customer reports repeated database timeouts during checkout.
Errors started after a deployment.
Revenue impact is high and customers are abandoning carts.
Suggested immediate action: restart service and roll back if needed.
""".strip()

print(ticket_text)

Customer reports repeated database timeouts during checkout.
Errors started after a deployment.
Revenue impact is high and customers are abandoning carts.
Suggested immediate action: restart service and roll back if needed.


In [5]:
# Baseline: run without schema enforcement and observe free-form output

loose_prompt = PromptTemplate(
    input_variables=["ticket_text"],
    template="Summarize this support ticket in 2-3 sentences:\n\n{ticket_text}\n",
)

loose_chain = loose_prompt | model | StrOutputParser()

if model is None:
    print("Model not initialized. Skipping execution.")
else:
    print(loose_chain.invoke({"ticket_text": ticket_text}))

Customer reports repeated database timeouts during checkout that began after a recent deployment, causing high revenue impact and widespread cart abandonment. They recommend immediate remediation by restarting the affected service and rolling back the deployment if the restart does not resolve the issue.


In [6]:
# Build a self-documenting chain by embedding the schema instructions into the prompt

strict_prompt = PromptTemplate(
    input_variables=["ticket_text"],
    template=(
        "Extract structured data from the ticket.\n"
        "Return output strictly in the required schema.\n"
        "Do not add explanations.\n\n"
        "{format_instructions}\n\n"
        "Ticket:\n{ticket_text}\n"
    ),
).partial(format_instructions=format_instructions)

strict_chain = strict_prompt | model | parser

In [7]:
# Invoke the strict chain and inspect the typed output

if model is None:
    print("Model not initialized. Skipping execution.")
else:
    result = strict_chain.invoke({"ticket_text": ticket_text})
    print(result)
    print("Type:", type(result))

priority=<Priority.HIGH: 'High'> issue='Repeated database timeouts during checkout after deployment causing customers to abandon carts' action='Restart the service immediately; if timeouts persist, roll back the recent deployment'
Type: <class '__main__.Ticket'>


In [8]:
# Use typed fields directly in downstream logic with no string parsing

if model is None:
    print("Model not initialized. Skipping execution.")
else:
    if result.priority == Priority.HIGH:
        print("Escalate immediately.")
    elif result.priority == Priority.MEDIUM:
        print("Queue for next on-call review.")
    else:
        print("Log and handle in normal cycle.")

Escalate immediately.
